# DeepSeek-R1-Distill-Qwen-7B — Fine-Tuning on Financial Reasoning Dataset

This notebook fine-tunes **DeepSeek-R1-Distill-Qwen-7B** on the v2 financial reasoning dataset (3,498 rows) using QLoRA.

> **Key difference vs the Qwen and Llama notebooks:**
> DeepSeek-R1-Distill-Qwen does **not** use a separate `system` role. All system-level instructions are folded into the **user** message. We still call `tokenizer.apply_chat_template(...)` — just with `user` + `assistant` only, no `system`.

## What this notebook does (high level)
1. Loads the v2 dataset (with reasoning_target already substituted to use `<|end_of_sentence|>` as the stop token, or falls back to `<|END|>` substitution in cell 5)
2. Validates the dataset for known v1 issues (contradictions, citation leakage, missing sections)
3. Builds prompts using the model's native chat template — **user + assistant only, no system role**
4. Tokenizes with target-only loss masking (model only learns from reasoning_target, not the prompt)
5. Trains the model with QLoRA in 4-bit (memory efficient)
6. Runs a smoke test on 5 manual cases to verify the model behaves correctly
7. Saves checkpoints to persistent storage and exports a clean merged model
8. Evaluates on the test set with EM, F1, format compliance, and arithmetic accuracy metrics in full mode

## Key design decisions (justified for the GP paper)
- **Loss masking** so the model only learns the reasoning, not the prompt
- **Native chat template** so the model uses its trained user/assistant structure
- **System instructions folded into user message** — DeepSeek-R1-Distill was not trained with a separate system role; using one degrades behavior
- **Refusal instruction added at inference only**, NOT during training (no refusal examples exist in training data)
- **bf16 when GPU supports it**, fp16 otherwise (avoids fp16 instability on Ampere+ GPUs)
- **Target-only stop token** (`<|end_of_sentence|>`) used natively by DeepSeek's chat template

## Two run modes
- `RUN_MODE = "smoke"` — 50 train rows, 1 epoch (~5-10 min). Use this FIRST to verify everything works.
- `RUN_MODE = "full"` — full dataset, 2 epochs (~4-8 hours on Colab A100/L4). Use after smoke test passes.

## Comparison to the other two notebooks
| Model                          | Chat template? | Separate system message? |
| ------------------------------ | -------------: | -----------------------: |
| Qwen2.5-7B-Instruct            |            Yes |                      Yes |
| Llama-3.1-8B-Instruct          |            Yes |                      Yes |
| DeepSeek-R1-Distill-Qwen-7B    |            Yes |                   **No** |


## 0. Setup, Environment Detection, and Global Configuration

**What this cell does:** works in both **Colab** and **Kaggle**, installs the required libraries, mounts Google Drive when running in Colab, detects the GPU, and sets all model paths.

**Important:** leave `RUN_MODE = "smoke"` for the first run. After the manual smoke test looks good, change it to `RUN_MODE = "full"` and rerun from the top.

**Note on stop token:** DeepSeek-R1-Distill-Qwen uses `<|end_of_sentence|>` as its end-of-turn marker (this is `tokenizer.eos_token`). Cell 3 verifies it resolves to a real special token.


In [1]:
# ============================================================
# 0) Setup + Environment Detection + Global Config
# ============================================================

import os

# Detect environment: Colab, Kaggle, or local Jupyter.
IS_COLAB = False
try:
    from google.colab import drive
    IS_COLAB = True
    drive.mount('/content/drive')
except Exception:
    IS_COLAB = False

IS_KAGGLE = os.path.exists('/kaggle/input') or os.path.exists('/kaggle/working')

print("Environment:", "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local/Unknown")

# Install/upgrade required libraries.
%pip install -q transformers accelerate peft bitsandbytes datasets evaluate openpyxl sentencepiece scikit-learn safetensors

import re
import time
import math
import json
import random
import shutil
import inspect
import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Capability:", torch.cuda.get_device_capability(0))

# ─────────────────────────────────────────────
# MODEL CONFIG
# ─────────────────────────────────────────────
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
MODEL_TAG  = "deepseek_r1_distill_qwen_7b"

# DeepSeek-R1-Distill uses fullwidth Unicode for its end-of-sentence token:
#   <｜end▁of▁sentence｜>  (U+FF5C vertical bars, U+2581 underscores)
# NOT the ASCII version <|end_of_sentence|>.
# We set this to None here and resolve it from tokenizer.eos_token in Cell 3,
# which is the safest way to avoid copy/paste encoding mistakes.
MODEL_STOP_TOKEN = None  # filled in from tokenizer.eos_token in Cell 3

MAX_LENGTH = 1024

# RUN_MODE controls dataset size and which heavy cells run.
# "smoke" = 50 rows, 1 epoch, skips clean export/full eval/zip.
# "full"  = full dataset, 2 epochs, saves checkpoints + exports model.
RUN_MODE = "full"

# Optional: a real CPU load test for a 7B merged model can require 16GB+ RAM.
RUN_CPU_LOAD_TEST = False

# ─────────────────────────────────────────────
# PATHS — safe for Colab and Kaggle
# ─────────────────────────────────────────────
if IS_COLAB:
    # Put the CSV in MyDrive root, or edit this path.
    DATA_FILE = f"/content/drive/MyDrive/tatqa_reasoning_v2_deepseek.csv"
    BASE_DIR  = f"/content/drive/MyDrive/gp2_training/{MODEL_TAG}"
elif IS_KAGGLE:
    # IMPORTANT: change YOUR_DATASET_NAME to the Kaggle dataset slug that contains the CSV.
    DATA_FILE = f"/kaggle/input/YOUR_DATASET_NAME/tatqa_reasoning_v2_{MODEL_TAG}.csv"
    BASE_DIR  = f"/kaggle/working/gp2_training/{MODEL_TAG}"
else:
    DATA_FILE = f"./tatqa_reasoning_v2_{MODEL_TAG}.csv"
    BASE_DIR  = f"./gp2_training/{MODEL_TAG}"

PERSISTENT_DIR = BASE_DIR
os.makedirs(PERSISTENT_DIR, exist_ok=True)

if RUN_MODE == "smoke":
    OUTPUT_DIR        = f"{PERSISTENT_DIR}/{MODEL_TAG}_smoke_outputs"
    ADAPTER_SAVE_PATH = f"{PERSISTENT_DIR}/{MODEL_TAG}_smoke_adapter"
    MERGED_SAVE_PATH  = f"{PERSISTENT_DIR}/{MODEL_TAG}_smoke_merged"
else:
    OUTPUT_DIR        = f"{PERSISTENT_DIR}/{MODEL_TAG}_full_outputs"   # checkpoints live here
    ADAPTER_SAVE_PATH = f"{PERSISTENT_DIR}/{MODEL_TAG}_full_adapter"
    MERGED_SAVE_PATH  = f"{PERSISTENT_DIR}/{MODEL_TAG}_full_merged"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("DATA_FILE:",        DATA_FILE)
print("PERSISTENT_DIR:",   PERSISTENT_DIR)
print("OUTPUT_DIR:",       OUTPUT_DIR)
print("MODEL_NAME:",       MODEL_NAME)
print("MODEL_STOP_TOKEN:", MODEL_STOP_TOKEN, "(will be resolved from tokenizer in Cell 3)")
print("MAX_LENGTH:",       MAX_LENGTH)
print("RUN_MODE:",         RUN_MODE)


Mounted at /content/drive
Environment: Colab
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
GPU Available: True
GPU Name: Tesla T4
GPU Capability: (7, 5)
DATA_FILE: /content/drive/MyDrive/tatqa_reasoning_v2_deepseek.csv
PERSISTENT_DIR: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b
OUTPUT_DIR: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_outputs
MODEL_NAME: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
MODEL_STOP_TOKEN: None (will be resolved from tokenizer in Cell 3)
MAX_LENGTH: 1024
RUN_MODE: full


## 1. Load Dataset and Verify Required Columns

**What this cell does:** loads the v2 dataset CSV, checks all required columns are present, fills any missing values, normalizes `answer_type` to lowercase.

**Expected output:** Shape `(3498, 9)` for the full dataset, distribution across answer types (span ~43%, arithmetic ~42%, multi-span ~12%, count ~2%), zero missing values after fill, and a preview of the first 3 rows.

**If something fails here:** Check that the CSV file path is correct and that the column names match exactly.


In [2]:
# ============================================================
# 1) Load Dataset + Required Columns Check
# ============================================================

if DATA_FILE.endswith(".xlsx"):
    df = pd.read_excel(DATA_FILE)
elif DATA_FILE.endswith(".csv"):
    df = pd.read_csv(DATA_FILE)
else:
    raise ValueError("Unsupported file type. Use .xlsx or .csv")

print("Dataset loaded ✅")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

required_columns = [
    "question",
    "answer",
    "derivation",
    "answer_type",
    "scale",
    "table_text",
    "relevant_paragraphs_text",
    "reasoning_target"
]

missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_columns].copy()

for col in required_columns:
    df[col] = df[col].fillna("").astype(str)

df["answer_type"] = df["answer_type"].str.strip().str.lower()

print("\nAnswer type distribution:")
print(df["answer_type"].value_counts(dropna=False))

print("\nMissing values after fill:")
print(df.isnull().sum())

display(df.head(3))


Dataset loaded ✅
Shape: (3498, 9)
Columns: ['question', 'answer', 'derivation', 'answer_type', 'answer_from', 'scale', 'table_text', 'relevant_paragraphs_text', 'reasoning_target']

Answer type distribution:
answer_type
span          1513
arithmetic    1467
multi-span     437
count           81
Name: count, dtype: int64

Missing values after fill:
question                    0
answer                      0
derivation                  0
answer_type                 0
scale                       0
table_text                  0
relevant_paragraphs_text    0
reasoning_target            0
dtype: int64


,question,answer,derivation,answer_type,scale,table_text,relevant_paragraphs_text,reasoning_target
0,What does Level 1 input in the fair value hier...,Observable inputs that reflect unadjusted quot...,,span,,"| | Fair value at December 31, 2019 | Level 1...",Level 1: Observable inputs that reflect unadju...,Reasoning & Calculations:\nValues used:\n- Lev...
1,What is the increase/ (decrease) in total oper...,-0.2,27.1-27.3,arithmetic,percent,"| | Year Ended December 31, | Year Ended Dece...",The 2019 operating expenses increased 3.9% com...,Reasoning & Calculations:\nValues used:\n- Tot...
2,In which year was Consumer Services larger?,2018,29.8>28.0,span,,| | 2019 | 2018 | |\n| Revenue | £m | £m | C...,"Revenue\nIn 2019, we saw another strong year o...",Reasoning & Calculations:\nValues used:\n- Con...


## 2. Dataset Validation Before Training

**What this cell does:** runs the same automated audit we used during dataset generation. Checks for: empty reasoning_targets, missing stop tokens, missing sections, citation leakage, arithmetic contradictions, and Analysis section content rules.

**Note:** the validation accepts **either** the model-native stop token `<|end_of_sentence|>` **or** the placeholder `<|END|>`. Cell 5 standardizes everything to `MODEL_STOP_TOKEN` before training. So if your dataset was generated with the generic `<|END|>` placeholder, this cell will still pass.

**Expected output for v2 dataset:** all critical failures should be 0. If any are non-zero, the dataset is corrupted and training should not proceed.


In [3]:
# ============================================================
# 2) Dataset Validation Before Training
# ============================================================

# This validation runs BEFORE <|END|> is replaced with the model-native stop token,
# so it accepts either token here. Cell 5 still standardizes targets to MODEL_STOP_TOKEN.
def contains_citation_leakage(text):
    text = str(text).lower()
    leakage_patterns = [
        "according to",
        "based on the source",
        "based on the table",
        "from the table",
        "from text",
        "source:",
        "citation",
        "page:",
        "paragraph:"
    ]
    return any(p in text for p in leakage_patterns)

def has_section(text, section):
    return section in str(text)

def has_stop_token(text):
    text = str(text)
    # Accept either the model-native stop token (set in Cell 3) or the
    # generic <|END|> placeholder. If MODEL_STOP_TOKEN hasn't been resolved
    # yet (Cell 3 not run), fall back to checking <|END|> only.
    if MODEL_STOP_TOKEN is not None and MODEL_STOP_TOKEN in text:
        return True
    return "<|END|>" in text

validation_report = {}

validation_report["empty_reasoning_target"] = int((df["reasoning_target"].str.strip() == "").sum())
validation_report["missing_stop_token"]     = int((~df["reasoning_target"].apply(has_stop_token)).sum())
validation_report["missing_values_used"]    = int((~df["reasoning_target"].apply(lambda t: has_section(t, "Values used:"))).sum())
validation_report["missing_calculation"]    = int((~df["reasoning_target"].apply(lambda t: has_section(t, "Calculation:"))).sum())
validation_report["missing_analysis"]       = int((~df["reasoning_target"].apply(lambda t: has_section(t, "Analysis:"))).sum())
validation_report["missing_final_answer"]   = int((~df["reasoning_target"].apply(lambda t: has_section(t, "Final Answer:"))).sum())
validation_report["citation_leakage"]       = int(df["reasoning_target"].apply(contains_citation_leakage).sum())

arithmetic_mask = df["answer_type"].eq("arithmetic")
non_arithmetic_mask = ~arithmetic_mask

validation_report["arithmetic_no_numerical_calculation_required"] = int(
    df.loc[arithmetic_mask, "reasoning_target"]
      .str.contains("No numerical calculation required", case=False, regex=False)
      .sum()
)

validation_report["non_arithmetic_with_real_analysis"] = int(
    df.loc[non_arithmetic_mask, "reasoning_target"]
      .apply(lambda t: "Analysis:" in t and "No financial analysis required" not in t)
      .sum()
)

print("===== DATASET VALIDATION REPORT =====")
for k, v in validation_report.items():
    print(f"{k}: {v}")

if any(v > 0 for v in validation_report.values()):
    print("\n⚠️ WARNING: Dataset has validation issues. Fix before training.")
else:
    print("\nDataset validation passed ✅")


===== DATASET VALIDATION REPORT =====
empty_reasoning_target: 0
missing_stop_token: 3498
missing_values_used: 0
missing_calculation: 0
missing_analysis: 0
missing_final_answer: 0
citation_leakage: 3
arithmetic_no_numerical_calculation_required: 0
non_arithmetic_with_real_analysis: 0

⚠️ WARNING: Dataset has validation issues. Fix before training.


## 3. Load Tokenizer + Verify Stop Token

**What this cell does:** loads the DeepSeek-R1-Distill-Qwen-7B tokenizer and verifies that `<|end_of_sentence|>` resolves to a real special token. This is critical — if the stop token isn't recognized, the model will never stop generating at inference time.

**Note:** DeepSeek's tokenizer typically reports `tokenizer.eos_token == '<|end_of_sentence|>'`, so `STOP_TOKEN_ID == tokenizer.eos_token_id`. That's expected and correct.

**Expected output:** stop token ID is a positive integer (not None, not unk_token_id). Pad token is set to eos_token if not already defined.


In [4]:
# ============================================================
# 3) Load Tokenizer + Verify Stop Token
# (loaded BEFORE prompt building because chat template needs it)
# ============================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# DeepSeek-R1-Distill-Qwen uses fullwidth Unicode chars in its end-of-sentence
# token (e.g. <｜end▁of▁sentence｜>), not ASCII. Hardcoding the literal string
# from notes/screenshots is fragile because pipes and underscores look identical
# in many fonts but differ in Unicode codepoints. The robust thing is to read
# the actual EOS token from the tokenizer and use that.
MODEL_STOP_TOKEN = tokenizer.eos_token
STOP_TOKEN_ID    = tokenizer.eos_token_id

print("Tokenizer loaded ✅")
print("Pad token:",         tokenizer.pad_token)
print("Pad token id:",      tokenizer.pad_token_id)
print("EOS token:",         tokenizer.eos_token)
print("EOS token id:",      tokenizer.eos_token_id)
print("MODEL_STOP_TOKEN:",  repr(MODEL_STOP_TOKEN))
print("STOP_TOKEN_ID:",     STOP_TOKEN_ID)

# Sanity checks
if STOP_TOKEN_ID is None or STOP_TOKEN_ID == tokenizer.unk_token_id:
    raise ValueError(
        f"Stop token '{MODEL_STOP_TOKEN}' did not resolve to a real special token. "
        f"This shouldn't happen — tokenizer.eos_token_id should always be valid."
    )

# Cross-check: the resolved token should round-trip cleanly
round_trip_id = tokenizer.convert_tokens_to_ids(MODEL_STOP_TOKEN)
if round_trip_id != STOP_TOKEN_ID:
    raise ValueError(
        f"Round-trip check failed: convert_tokens_to_ids({MODEL_STOP_TOKEN!r}) = {round_trip_id} "
        f"but tokenizer.eos_token_id = {STOP_TOKEN_ID}. Tokenizer state is inconsistent."
    )

print(f"\n✓ Stop token verified for {MODEL_NAME}")
print(f"  Resolved from tokenizer.eos_token: {MODEL_STOP_TOKEN!r}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded ✅
Pad token: <｜end▁of▁sentence｜>
Pad token id: 151643
EOS token: <｜end▁of▁sentence｜>
EOS token id: 151643
MODEL_STOP_TOKEN: '<｜end▁of▁sentence｜>'
STOP_TOKEN_ID: 151643

✓ Stop token verified for deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
  Resolved from tokenizer.eos_token: '<｜end▁of▁sentence｜>'


## 4. Define System Instructions and Prompt Builders (DeepSeek style)

**🔑 This is the cell that differs the most from Qwen and Llama notebooks.**

For Qwen and Llama we used:
```python
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_message},
]
```

For **DeepSeek-R1-Distill-Qwen** we use:
```python
messages = [
    {"role": "user", "content": full_user_prompt},  # instructions + question + context all together
]
```

**Why:** DeepSeek-R1-Distill-Qwen was distilled from DeepSeek-R1, which was trained without a meaningful separate system role. Using a system role degrades adherence. The cleaner setup is to fold the instructions into the user message and let `apply_chat_template(...)` handle the rest.

**What stays the same:**
- We still call `tokenizer.apply_chat_template(...)` — just with `user` + `assistant`, no `system`.
- The training instructions are identical to Qwen/Llama (same structure, same sections, same refusal rule at inference only).
- Loss masking in cell 8 still masks everything except the assistant target.

**Expected output:** confirmation that builders are defined. No data is processed yet.


In [5]:
# ============================================================
# 4) Define Instructions + Prompt Builders Using Chat Template
#    DeepSeek-R1-Distill-Qwen — NO separate system role.
#    All instructions go inside the user message.
# ============================================================

# Training instructions — describe the structure the model should produce.
# NOTE: refusal instruction is intentionally excluded here because the
# training data has zero refusal examples. The base model's RLHF refusal
# capability handles refusal at inference when given the inference instructions.
TRAINING_INSTRUCTIONS = """You are an expert financial analyst answering questions about company reports.

Answer using only the provided context. Structure your response with these sections:

Reasoning & Calculations:
Values used: List each value from the context used to answer.
Calculation: Show the formula and result. If no calculation is needed, write: No numerical calculation required.
Analysis: For arithmetic questions, write 1-2 sentences interpreting the numeric result. For all other questions, write: No financial analysis required.
Final Answer: One sentence that directly answers the question. Include the scale (thousand, million, percent) when applicable."""

# Inference instructions — same as training plus the refusal instruction.
# Used at inference time only.
INFERENCE_INSTRUCTIONS = TRAINING_INSTRUCTIONS + """

If the context does not contain enough information to answer, respond with exactly:
"I do not have enough information to answer this based on the provided document."
"""


def build_question_and_context(row_or_dict):
    """Build the question + context block (no instructions yet)."""
    if isinstance(row_or_dict, dict):
        question          = row_or_dict.get("question", "")
        table_context     = row_or_dict.get("table_text", "").strip()
        paragraph_context = row_or_dict.get("relevant_paragraphs_text", "").strip()
    else:
        question          = row_or_dict["question"]
        table_context     = row_or_dict["table_text"].strip()
        paragraph_context = row_or_dict["relevant_paragraphs_text"].strip()

    context_parts = []
    if table_context:
        context_parts.append(f"Table Context:\n{table_context}")
    if paragraph_context:
        context_parts.append(f"Paragraph Context:\n{paragraph_context}")

    context_block = "\n\n".join(context_parts) if context_parts else "No context provided."

    return f"""Question:
{question}

{context_block}"""


def build_full_user_message(row_or_dict, instructions):
    """Fold instructions + question + context into a single user message.
    This is the DeepSeek-style prompt."""
    qc = build_question_and_context(row_or_dict)
    return f"{instructions}\n\n{qc}"


def build_training_prompt(row, instructions=TRAINING_INSTRUCTIONS):
    """Build the training prompt using the model's native chat template.
    DeepSeek style: ONLY a user turn, no system turn. The assistant response
    (target_text) is appended separately during tokenization.
    add_generation_prompt=True ensures the prompt ends with the assistant
    turn marker so the target naturally follows."""
    user_message = build_full_user_message(row, instructions)
    messages = [
        {"role": "user", "content": user_message},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def build_inference_prompt(question, table_context="", paragraph_context=""):
    """Build the inference prompt with the refusal instruction included.
    DeepSeek style: ONLY a user turn."""
    user_message = build_full_user_message(
        {
            "question": question,
            "table_text": table_context,
            "relevant_paragraphs_text": paragraph_context
        },
        INFERENCE_INSTRUCTIONS
    )
    messages = [
        {"role": "user", "content": user_message},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


print("Prompt builders defined ✅ (DeepSeek style — no separate system role)")


Prompt builders defined ✅ (DeepSeek style — no separate system role)


## 5. Apply Prompt Template to All Rows

**What this cell does:** uses `build_training_prompt()` to produce `input_text` for every row in the dataset. Also runs a defensive substitution of `<|END|>` to the model's stop token in case the dataset wasn't pre-substituted. Prints the first sample to verify the chat template structure looks correct.

**Expected output for DeepSeek:** sample `input_text` should contain DeepSeek's chat template markers (typically starting with `<｜begin▁of▁sentence｜>` and `<｜User｜>` / `<｜Assistant｜>` separators). Stop token count in target should equal the number of rows.

**Sanity check:** the formatted prompt should NOT contain a `system` block — only the user message followed by the assistant turn marker.


In [6]:
# ============================================================
# 5) Apply Prompt Template + Verify Structure
# ============================================================

# Defensive: ensure all targets use the model-native stop token.
# If the dataset was already pre-substituted, this is a no-op.
df["target_text"] = df["reasoning_target"].str.replace("<|END|>", MODEL_STOP_TOKEN, regex=False)

# Build the training prompts using the chat template
df["input_text"] = df.apply(build_training_prompt, axis=1)

print("Prompts built using chat template ✅")
print("Rows:", len(df))

print("\n===== SAMPLE INPUT (chat template formatted) =====")
print(df.loc[0, "input_text"][:2000])

print("\n===== SAMPLE TARGET =====")
print(df.loc[0, "target_text"][:1500])

print("\nStop token count in target:")
print(df["target_text"].str.contains(MODEL_STOP_TOKEN, regex=False).sum())

# Sanity check: chat template markers should be present and there should be NO system block
sample = df.loc[0, "input_text"]
print("\n===== CHAT TEMPLATE SANITY CHECK =====")
print(f"First 200 chars: {sample[:200]}")
print(f"\nLast 200 chars: {sample[-200:]}")

# DeepSeek-specific check: no system role marker should appear
# (DeepSeek tokenizer doesn't emit a 'system' header when no system role is provided)
print("\nDeepSeek-style check (no separate system role):")
print(f"  Contains 'system' literal? {('system' in sample.lower())}  (informational only — DeepSeek may not use the word at all)")


Prompts built using chat template ✅
Rows: 3498

===== SAMPLE INPUT (chat template formatted) =====
<｜begin▁of▁sentence｜><｜User｜>You are an expert financial analyst answering questions about company reports.

Answer using only the provided context. Structure your response with these sections:

Reasoning & Calculations:
Values used: List each value from the context used to answer.
Calculation: Show the formula and result. If no calculation is needed, write: No numerical calculation required.
Analysis: For arithmetic questions, write 1-2 sentences interpreting the numeric result. For all other questions, write: No financial analysis required.
Final Answer: One sentence that directly answers the question. Include the scale (thousand, million, percent) when applicable.

Question:
What does Level 1 input in the fair value hierarchy refer to?

Table Context:
|  | Fair value at December 31, 2019 | Level 1 | Level 2 | Level 3 |
| Cash equivalents: |  |  |  |  |
| Money market funds | $297,311 |

## 6. Stratified Train / Validation / Test Split

**What this cell does:** splits the dataset 80/10/10 with stratification by `answer_type` so each split preserves the natural distribution. In smoke mode, only 50 train rows and 30 val rows are used. Also runs a leakage check to confirm no overlap between splits.

**Expected output:** Train ~2,798 rows (smoke: 50), Val ~350 rows (smoke: 30), Test ~350 rows. All three leakage intersections should be 0.


In [7]:
# ============================================================
# 6) Stratified Train / Validation / Test Split
# ============================================================

from sklearn.model_selection import train_test_split

df = df[
    (df["input_text"].str.strip() != "") &
    (df["target_text"].str.strip() != "")
].copy()

df.reset_index(drop=True, inplace=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["answer_type"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["answer_type"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

if RUN_MODE == "smoke":
    smoke_df, _ = train_test_split(
        train_df,
        train_size=min(50, len(train_df)),
        random_state=SEED,
        stratify=train_df["answer_type"]
    )
    train_df_used = smoke_df.reset_index(drop=True)
    val_df_used = val_df.sample(min(30, len(val_df)), random_state=SEED).reset_index(drop=True)
else:
    train_df_used = train_df.copy()
    val_df_used   = val_df.copy()

print("===== SPLIT SIZES =====")
print("Train used:",      train_df_used.shape)
print("Validation used:", val_df_used.shape)
print("Test:",            test_df.shape)

print("\nTrain distribution:")
print(train_df_used["answer_type"].value_counts(normalize=True).round(4))

print("\nValidation distribution:")
print(val_df_used["answer_type"].value_counts(normalize=True).round(4))

print("\nTest distribution:")
print(test_df["answer_type"].value_counts(normalize=True).round(4))

# leakage check
train_pairs = set(zip(train_df_used["question"], train_df_used["table_text"], train_df_used["relevant_paragraphs_text"]))
val_pairs   = set(zip(val_df_used["question"],   val_df_used["table_text"],   val_df_used["relevant_paragraphs_text"]))
test_pairs  = set(zip(test_df["question"],       test_df["table_text"],       test_df["relevant_paragraphs_text"]))

print("\n===== LEAKAGE CHECK =====")
print("Train ∩ Validation:", len(train_pairs & val_pairs))
print("Train ∩ Test:",       len(train_pairs & test_pairs))
print("Validation ∩ Test:",  len(val_pairs & test_pairs))


===== SPLIT SIZES =====
Train used: (2798, 10)
Validation used: (350, 10)
Test: (350, 10)

Train distribution:
answer_type
span          0.4325
arithmetic    0.4192
multi-span    0.1251
count         0.0232
Name: proportion, dtype: float64

Validation distribution:
answer_type
span          0.4343
arithmetic    0.4200
multi-span    0.1229
count         0.0229
Name: proportion, dtype: float64

Test distribution:
answer_type
span          0.4314
arithmetic    0.4200
multi-span    0.1257
count         0.0229
Name: proportion, dtype: float64

===== LEAKAGE CHECK =====
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


## 7. Token Length Statistics

**What this cell does:** computes prompt and target token lengths across the training set so you can verify nothing exceeds MAX_LENGTH (1024).

**Note for DeepSeek:** because we fold the instructions into the user message, DeepSeek prompts are slightly **longer** in tokens than Qwen/Llama prompts (which split the same text across system + user). Expect prompt_len p99 to be ~50-100 tokens higher than the other two notebooks. Total length should still be safely under 1024 for the v2 dataset.

**Expected output:** total_len p99 should be well under 1024. If "Samples longer than MAX_LENGTH" is non-zero, you may need to increase MAX_LENGTH or the truncation logic in the next cell will silently drop context.


In [8]:
# ============================================================
# 7) Token Length Stats
# ============================================================

def token_len_prompt_target(row):
    prompt_ids = tokenizer(row["input_text"],  add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(row["target_text"], add_special_tokens=False)["input_ids"]
    return len(prompt_ids), len(target_ids), len(prompt_ids) + len(target_ids)

lengths = train_df_used.apply(token_len_prompt_target, axis=1, result_type="expand")
lengths.columns = ["prompt_len", "target_len", "total_len"]

print("===== TOKEN LENGTH STATS =====")
display(lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

too_long_ratio = (lengths["total_len"] > MAX_LENGTH).mean() * 100
print(f"Samples longer than MAX_LENGTH: {too_long_ratio:.2f}%")

# Hard assertion — if any target exceeds MAX_LENGTH on its own, the
# tokenization step would wipe the prompt entirely and produce corrupted training data.
max_target = lengths["target_len"].max()
assert max_target < MAX_LENGTH, f"A target is {max_target} tokens, exceeds MAX_LENGTH={MAX_LENGTH}. Increase MAX_LENGTH or truncate targets."
print(f"\n✓ Max target length ({max_target}) is safely under MAX_LENGTH ({MAX_LENGTH})")


===== TOKEN LENGTH STATS =====


,prompt_len,target_len,total_len
count,2798.000000,2798.000000,2798.000000
mean,548.138670,131.853467,679.992137
std,270.633492,55.291783,285.798986
min,218.000000,48.000000,277.000000
50%,477.000000,124.000000,608.500000
75%,640.000000,169.000000,784.750000
90%,842.000000,204.000000,1015.300000
95%,1074.150000,230.000000,1230.450000
99%,1546.180000,281.030000,1710.210000
max,2950.000000,655.000000,3097.000000


Samples longer than MAX_LENGTH: 9.86%

✓ Max target length (655) is safely under MAX_LENGTH (1024)


## 8. Tokenization with Target-Only Loss Masking

**What this cell does:** tokenizes each (prompt, target) pair and creates `labels` with `-100` everywhere except on the target tokens. This means the model only computes loss on the target — it does NOT learn to reproduce the prompt.

**This logic is identical to the Qwen and Llama notebooks** — the masking does not depend on whether a system role exists. It just masks everything before the assistant target.

**Expected output:** sample `Supervised target tokens` should be ~100-300 (the actual reasoning_target length), `Ignored prompt/pad tokens` should be the rest. A decoded preview of the training sample shows what the model actually sees.


In [9]:
# ============================================================
# 8) Tokenization with Target-Only Loss Masking
# ============================================================

from datasets import Dataset, DatasetDict

def tokenize_with_masking(row):
    prompt_ids = tokenizer(row["input_text"],  add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(row["target_text"], add_special_tokens=False)["input_ids"]

    # If target alone exceeds MAX_LENGTH, truncate it (rare, asserted impossible above)
    if len(target_ids) >= MAX_LENGTH:
        target_ids = target_ids[:MAX_LENGTH]
        prompt_ids = []
    else:
        # Truncate prompt from the LEFT if combined exceeds MAX_LENGTH
        # (preserves the part closest to the assistant response)
        max_prompt_len = MAX_LENGTH - len(target_ids)
        prompt_ids = prompt_ids[-max_prompt_len:]

    input_ids      = prompt_ids + target_ids
    labels         = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    pad_len = MAX_LENGTH - len(input_ids)
    input_ids      = input_ids      + [tokenizer.pad_token_id] * pad_len
    labels         = labels         + [-100] * pad_len
    attention_mask = attention_mask + [0]    * pad_len

    return {
        "input_ids":      input_ids,
        "labels":         labels,
        "attention_mask": attention_mask
    }

train_dataset = Dataset.from_pandas(
    train_df_used[["input_text", "target_text", "answer_type", "question", "answer"]],
    preserve_index=False
)
val_dataset = Dataset.from_pandas(
    val_df_used[["input_text", "target_text", "answer_type", "question", "answer"]],
    preserve_index=False
)
test_dataset = Dataset.from_pandas(
    test_df[["input_text", "target_text", "answer_type", "question", "answer"]],
    preserve_index=False
)

tokenized_train = train_dataset.map(tokenize_with_masking, remove_columns=train_dataset.column_names)
tokenized_val   = val_dataset.map(tokenize_with_masking,   remove_columns=val_dataset.column_names)
tokenized_test  = test_dataset.map(tokenize_with_masking,  remove_columns=test_dataset.column_names)

print("Tokenization with masking completed ✅")
print(tokenized_train)

sample = tokenized_train[0]
print("\nSample lengths:")
print("input_ids:",      len(sample["input_ids"]))
print("labels:",         len(sample["labels"]))
print("attention_mask:", len(sample["attention_mask"]))

label_count   = sum(1 for x in sample["labels"] if x != -100)
ignored_count = sum(1 for x in sample["labels"] if x == -100)
print("Supervised target tokens:", label_count)
print("Ignored prompt/pad tokens:", ignored_count)

decoded_visible = tokenizer.decode(
    [x for x in sample["input_ids"] if x != tokenizer.pad_token_id],
    skip_special_tokens=False
)
print("\nDecoded training sample preview (first 2500 chars):")
print(decoded_visible[:2500])


Map:   0%|          | 0/2798 [00:00<?, ? examples/s]

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

Tokenization with masking completed ✅
Dataset({
    features: ['input_ids', 'labels', 'attention_mask'],
    num_rows: 2798
})

Sample lengths:
input_ids: 1024
labels: 1024
attention_mask: 1024
Supervised target tokens: 161
Ignored prompt/pad tokens: 863

Decoded training sample preview (first 2500 chars):
<｜begin▁of▁sentence｜><｜User｜>You are an expert financial analyst answering questions about company reports.

Answer using only the provided context. Structure your response with these sections:

Reasoning & Calculations:
Values used: List each value from the context used to answer.
Calculation: Show the formula and result. If no calculation is needed, write: No numerical calculation required.
Analysis: For arithmetic questions, write 1-2 sentences interpreting the numeric result. For all other questions, write: No financial analysis required.
Final Answer: One sentence that directly answers the question. Include the scale (thousand, million, percent) when applicable.

Question:
What 

## 9. Load DeepSeek-R1-Distill-Qwen-7B with 4-bit Quantization + QLoRA

**What this cell does:** loads DeepSeek-R1-Distill-Qwen-7B in 4-bit (NF4 quantization) for memory efficiency, enables gradient checkpointing, and adds LoRA adapters on attention and MLP projections (q_proj, k_proj, v_proj, o_proj, up_proj, down_proj, gate_proj).

**Why these target modules:** DeepSeek-R1-Distill-Qwen has the same Qwen-2.5 architecture (it's a Qwen-2.5 base distilled from DeepSeek-R1), so the LoRA target modules are identical to the Qwen notebook.

**Why QLoRA:** training a 7B parameter model in full precision needs >40GB VRAM. 4-bit quantization + LoRA reduces this to ~10-15GB, fits on Colab T4/L4.

**Expected output:** trainable params should be ~40M-50M, which is ~0.5-0.7% of total parameters. Compute dtype should be `torch.bfloat16` on Ampere+ GPUs (A100, L4) or `torch.float16` on older GPUs (T4).


In [10]:
# ============================================================
# 9) Load Model with QLoRA
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Use bf16 on Ampere+ GPUs (capability >= 8.0), fp16 on older GPUs
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True

print("compute_dtype:", compute_dtype)
print("use_bf16:",      use_bf16)
print("use_fp16:",      use_fp16)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# Same target modules as Qwen — DeepSeek-R1-Distill-Qwen-7B uses the Qwen-2.5 architecture
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "up_proj", "down_proj", "gate_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

def print_trainable_parameters(m):
    trainable_params = 0
    all_params = 0
    for _, param in m.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"Trainable params: {trainable_params:,}")
    print(f"All params:       {all_params:,}")
    print(f"Trainable %:      {100 * trainable_params / all_params:.4f}%")
    return trainable_params, all_params, 100 * trainable_params / all_params

trainable_params, all_params, trainable_pct = print_trainable_parameters(model)


compute_dtype: torch.float16
use_bf16: False
use_fp16: True


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Trainable params: 40,370,176
All params:       4,393,342,464
Trainable %:      0.9189%


## 10. Training Arguments + Trainer Setup

**What this cell does:** builds `TrainingArguments` and the `Trainer` while keeping our target-only labels intact with `default_data_collator`.

**Important fixes included:**
- Checkpoints save every `50` steps in full mode, so a crash does not waste the whole run.
- Smoke mode stays lightweight.
- `load_best_model_at_end=True` only when eval/save strategies match.

**Expected output:** step counts, checkpoint interval, and `Trainer initialized ✅`.


In [11]:
# ============================================================
# 10) TrainingArguments + Trainer
# ============================================================

from transformers import TrainingArguments, Trainer, default_data_collator

num_epochs = 1 if RUN_MODE == "smoke" else 2

per_device_train_batch_size = 4
per_device_eval_batch_size  = 4
gradient_accumulation_steps = 4
effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps

train_samples        = len(tokenized_train)
steps_per_epoch      = math.ceil(train_samples / effective_batch_size)
total_training_steps = steps_per_epoch * num_epochs
warmup_steps         = max(1, int(total_training_steps * 0.03))

# Save checkpoints often enough that a Colab/Kaggle crash does not waste hours.
save_steps = 10 if RUN_MODE == "smoke" else 50
eval_steps = save_steps

print("Train samples:",          train_samples)
print("Effective batch size:",   effective_batch_size)
print("Steps per epoch:",        steps_per_epoch)
print("Total training steps:",   total_training_steps)
print("Warmup steps:",           warmup_steps)
print("Checkpoint save steps:",  save_steps)
print("Output/checkpoint dir:",  OUTPUT_DIR)

# Handle different transformers versions
sig = inspect.signature(TrainingArguments.__init__)
supported_args = set(sig.parameters.keys())

args_dict = {
    "output_dir":                  OUTPUT_DIR,
    "overwrite_output_dir":        False,
    "num_train_epochs":            num_epochs,
    "per_device_train_batch_size": per_device_train_batch_size,
    "per_device_eval_batch_size":  per_device_eval_batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "learning_rate":               2e-4,
    "weight_decay":                0.01,
    "warmup_steps":                warmup_steps,
    "lr_scheduler_type":           "cosine",
    "fp16":                        use_fp16,
    "bf16":                        use_bf16,
    "logging_steps":               10 if RUN_MODE == "smoke" else 20,
    "save_strategy":               "steps",
    "save_steps":                  save_steps,
    "save_total_limit":            5,

    "load_best_model_at_end":      True,
    "metric_for_best_model":       "eval_loss",
    "greater_is_better":           False,

    "report_to":                   "none",
    "seed":                        SEED
}

# Match evaluation strategy to save strategy.
if "eval_strategy" in supported_args:
    args_dict["eval_strategy"] = "steps"
elif "evaluation_strategy" in supported_args:
    args_dict["evaluation_strategy"] = "steps"

if "eval_steps" in supported_args:
    args_dict["eval_steps"] = eval_steps

if "save_safetensors" in supported_args:
    args_dict["save_safetensors"] = True

args_dict = {k: v for k, v in args_dict.items() if k in supported_args}

training_args = TrainingArguments(**args_dict)

print("TrainingArguments created ✅")

trainer_sig = inspect.signature(Trainer.__init__)
trainer_supported_args = set(trainer_sig.parameters.keys())

trainer_kwargs = {
    "model":          model,
    "args":           training_args,
    "train_dataset":  tokenized_train,
    "eval_dataset":   tokenized_val,
    "data_collator":  default_data_collator
}

if "tokenizer" in trainer_supported_args:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer initialized ✅")


Train samples: 2798
Effective batch size: 16
Steps per epoch: 175
Total training steps: 350
Warmup steps: 10
Checkpoint save steps: 50
Output/checkpoint dir: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_outputs
TrainingArguments created ✅
Trainer initialized ✅


## 11. Train + Save Logs

**What this cell does:** starts training and automatically resumes from the latest checkpoint in `OUTPUT_DIR` if the notebook/runtime crashes and you rerun it.

**Expected output:** if no checkpoint exists, it starts fresh. If a checkpoint exists, it prints the checkpoint path and resumes from there. Logs are saved to the persistent folder, not temporary session storage.


In [12]:
# ============================================================
# 11) Train + Save Training Logs + Auto-Resume
# ============================================================

def get_latest_checkpoint(output_dir):
    """Return the latest checkpoint path inside output_dir, or None."""
    if not os.path.exists(output_dir):
        return None

    checkpoints = []
    for name in os.listdir(output_dir):
        full_path = os.path.join(output_dir, name)
        if os.path.isdir(full_path) and name.startswith("checkpoint-"):
            try:
                step = int(name.split("-")[-1])
                checkpoints.append((step, full_path))
            except ValueError:
                pass

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1]

resume_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if resume_checkpoint:
    print(f"Found existing checkpoint: {resume_checkpoint}")
    print("Resuming training from this checkpoint...")
else:
    print("No existing checkpoint found. Starting fresh training run...")

start_time = time.time()
print("Training started...")

if resume_checkpoint:
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
else:
    train_result = trainer.train()

end_time = time.time()
training_time_seconds = end_time - start_time
training_time_minutes = training_time_seconds / 60

print("\nTraining completed ✅")
print(f"Training time seconds: {training_time_seconds:.2f}")
print(f"Training time minutes: {training_time_minutes:.2f}")

print("\nTrain result metrics:")
print(train_result.metrics)

train_metrics_df = pd.DataFrame([train_result.metrics])
display(train_metrics_df)

log_history_df = pd.DataFrame(trainer.state.log_history)
display(log_history_df.tail(10))

training_log_path     = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_training_log_history.csv"
training_metrics_path = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_training_metrics.csv"

log_history_df.to_csv(training_log_path, index=False)
train_metrics_df.to_csv(training_metrics_path, index=False)

print("Logs saved to persistent storage:")
print(training_log_path)
print(training_metrics_path)


Found existing checkpoint: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_outputs/checkpoint-350
Resuming training from this checkpoint...
Training started...


Step,Training Loss,Validation Loss



Training completed ✅
Training time seconds: 30.17
Training time minutes: 0.50

Train result metrics:
{'train_runtime': 0.2066, 'train_samples_per_second': 27081.007, 'train_steps_per_second': 1693.773, 'total_flos': 2.444887829910651e+17, 'train_loss': 0.0, 'epoch': 2.0}


,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss,epoch
0,0.2066,27081.007,1693.773,2.444888e+17,0.0,2.0


,epoch,grad_norm,learning_rate,loss,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
15,1.371429,0.171814,4.814462e-05,0.112526,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,1.428571,NaN,NaN,NaN,250,0.129652,45.2855,7.729,1.943,NaN,NaN,NaN,NaN,NaN
17,1.485714,0.186466,3.331614e-05,0.099552,260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,1.600000,0.236282,2.075851e-05,0.108627,280,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,1.714286,0.196315,1.089935e-05,0.113069,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,1.714286,NaN,NaN,NaN,300,0.128018,45.2724,7.731,1.944,NaN,NaN,NaN,NaN,NaN
21,1.828571,0.184435,4.074402e-06,0.100997,320,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,1.942857,0.204661,5.160875e-07,0.113122,340,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,2.000000,NaN,NaN,NaN,350,0.127882,45.2435,7.736,1.945,NaN,NaN,NaN,NaN,NaN
24,2.000000,NaN,NaN,NaN,350,NaN,NaN,NaN,NaN,0.2066,27081.007,1693.773,2.444888e+17,0.0


Logs saved to persistent storage:
/content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_training_log_history.csv
/content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_training_metrics.csv


## 12. Manual Smoke Test on 5 Test Cases

**What this cell does:** runs 5 hand-crafted inference cases covering all question types (arithmetic, span, multi-span, count, insufficient context). Uses the INFERENCE prompt (with refusal instruction).

**What to check in the output:**
- ✅ Each output ends cleanly with the stop token (no mid-word truncation)
- ✅ Arithmetic case has Analysis section with real interpretation
- ✅ Span/multi-span/count cases have `Analysis: No financial analysis required.`
- ✅ All outputs have Values used / Calculation / Final Answer sections
- ✅ Insufficient context case produces the exact refusal sentence
- ❌ No prompt leakage (no "Human:", "Question:", "Table Context:" appearing in output)
- ❌ No contradictions (no "No numerical calculation required" followed by a calculation)

**DeepSeek-specific note:** R1-Distill models tend to emit `<think>...</think>` reasoning blocks before the final answer. Our `clean_generated_response()` strips those when present, so the displayed output is the structured response only. If you want to keep raw `<think>` blocks for analysis, set `STRIP_THINK_BLOCKS = False` below.

**If smoke test fails:** stop, debug, do not proceed to full training.


In [13]:
# ============================================================
# 12) Manual Smoke Test on 5 Test Cases
# ============================================================

model.eval()
model.config.use_cache = True

# DeepSeek R1 Distill emits <think>...</think> blocks. Strip by default for the
# structured output we trained on. Set to False if you want to keep them.
STRIP_THINK_BLOCKS = True

def clean_generated_response(text):
    text = str(text)
    if MODEL_STOP_TOKEN in text:
        text = text.split(MODEL_STOP_TOKEN)[0].strip()

    # DeepSeek-specific: strip <think>...</think> blocks if present
    if STRIP_THINK_BLOCKS:
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        # Also strip an unclosed <think> block if generation cut off inside it
        text = re.sub(r"<think>.*$", "", text, flags=re.DOTALL).strip()

    leakage_markers = ["Human:", "User:", "Assistant:", "Question:"]
    for marker in leakage_markers:
        if marker in text:
            text = text.split(marker)[0].strip()
    return text.strip()

def generate_response(prompt, max_new_tokens=400):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    eos_ids = [tokenizer.eos_token_id, STOP_TOKEN_ID]
    eos_ids = list({x for x in eos_ids if x is not None})

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id
        )

    full_text   = tokenizer.decode(outputs[0],          skip_special_tokens=False)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)

    if full_text.startswith(prompt_text):
        response = full_text[len(prompt_text):].strip()
    else:
        response = full_text

    return clean_generated_response(response)

manual_cases = [
    {
        "type": "Arithmetic",
        "question": "What is the percentage increase in net income from 2023 to 2024?",
        "table_context": """Row 1:  | 2024 | 2023
Row 2: Net income | $540 million | $450 million""",
        "paragraph_context": "The company reported stronger profitability in 2024 compared with the previous year."
    },
    {
        "type": "Span",
        "question": "What was the operating margin in 2024?",
        "table_context": """Row 1:  | 2024 | 2023
Row 2: Revenue | $2,100 million | $1,950 million
Row 3: Operating margin | 18.4% | 16.9%""",
        "paragraph_context": ""
    },
    {
        "type": "Multi-span",
        "question": "What were revenue and net income in 2024?",
        "table_context": """Row 1:  | 2024 | 2023
Row 2: Revenue | $2,100 million | $1,950 million
Row 3: Net income | $540 million | $450 million""",
        "paragraph_context": ""
    },
    {
        "type": "Count",
        "question": "How many regions are listed in the table?",
        "table_context": """Row 1: Region | 2024 Revenue
Row 2: Americas | $900 million
Row 3: Europe | $700 million
Row 4: Asia Pacific | $500 million""",
        "paragraph_context": ""
    },
    {
        "type": "Insufficient Context",
        "question": "What was the earnings per share in 2024?",
        "table_context": """Row 1:  | 2024 | 2023
Row 2: Revenue | $2,100 million | $1,950 million
Row 3: Net income | $540 million | $450 million""",
        "paragraph_context": ""
    }
]

for i, case in enumerate(manual_cases, start=1):
    prompt = build_inference_prompt(
        case["question"],
        case["table_context"],
        case["paragraph_context"]
    )
    pred = generate_response(prompt, max_new_tokens=400)

    print("=" * 120)
    print(f"CASE {i} | TYPE: {case['type']}")
    print("\nQUESTION:")
    print(case["question"])
    print("\nOUTPUT:")
    print(pred)
    print()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CASE 1 | TYPE: Arithmetic

QUESTION:
What is the percentage increase in net income from 2023 to 2024?

OUTPUT:
Reasoning & Calculations:
Values used:
- Net income 2024: $540 million
- Net income 2023: $450 million
Calculation:
(540 - 450) / 450 = 0.2
0.2 x 100 = 20 percent
Analysis:
Net income increased by 20 percent from 2023 to 2024.
Final Answer:
The percentage increase in net income from 2023 to 2024 is 20 percent.
<|end▁of▁sentence|>

CASE 2 | TYPE: Span

QUESTION:
What was the operating margin in 2024?

OUTPUT:
Reasoning & Calculations:
Values used:
- Operating margin 2024: 18.4%
Calculation:
No numerical calculation required.
Analysis:
No financial analysis required.
Final Answer:
The operating margin in 2024 was 18.4%.
<|end▁of▁sentence|>

CASE 3 | TYPE: Multi-span

QUESTION:
What were revenue and net income in 2024?

OUTPUT:
Reasoning & Calculations:
Values used:
- Revenue 2024: $2,100 million
- Net income 2024: $540 million
Calculation:
No numerical calculation required.
Anal

## 13. Save Adapter + Clean Merged Model Export

**What this cell does:**
- Skips during `RUN_MODE = "smoke"`.
- Saves the LoRA adapter.
- Merges LoRA into the base model for deployment.
- Removes `quantization_config` from the exported config so the merged model is not forced to load with bitsandbytes later.
- Saves the merged model using safetensors shards.

**Expected output in full mode:** adapter folder created, merged folder created, `config.json` has no `quantization_config`, and the merged folder contains `.safetensors` shards.


In [14]:
# ============================================================
# 13) Save Adapter + Clean Merged Model Export
# ============================================================

if RUN_MODE == "smoke":
    print("Skipped on smoke run — clean export only happens after full training.")
    print("If the manual smoke test looks correct, change RUN_MODE to 'full' and rerun from the top.")
else:
    print("Adapter path:",      ADAPTER_SAVE_PATH)
    print("Merged model path:", MERGED_SAVE_PATH)

    os.makedirs(ADAPTER_SAVE_PATH, exist_ok=True)
    os.makedirs(MERGED_SAVE_PATH, exist_ok=True)

    print("\nSaving LoRA adapter...")
    trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
    tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
    print("Adapter + tokenizer saved ✅")

    print("\nMerging LoRA into base model...")
    merged_model = model.merge_and_unload()
    print("LoRA merged ✅")

    # Critical fix: remove quantization metadata so CPU/non-bitsandbytes loading
    # does not fail later by trying to reload the merged model as 4-bit.
    if hasattr(merged_model.config, "quantization_config"):
        print("Removing quantization_config from merged model config ✅")
        delattr(merged_model.config, "quantization_config")

    if hasattr(merged_model, "generation_config") and hasattr(merged_model.generation_config, "quantization_config"):
        delattr(merged_model.generation_config, "quantization_config")

    print("\nSaving clean merged model with safetensors shards...")
    merged_model.save_pretrained(
        MERGED_SAVE_PATH,
        safe_serialization=True,
        max_shard_size="4GB"
    )
    tokenizer.save_pretrained(MERGED_SAVE_PATH)
    print("Merged model + tokenizer saved ✅")

    # Verify config is clean.
    config_path = os.path.join(MERGED_SAVE_PATH, "config.json")
    with open(config_path, "r", encoding="utf-8") as f:
        verify_config = json.load(f)

    has_quant = "quantization_config" in verify_config
    print(f"\nVerification — quantization_config in config.json: {has_quant}")
    if has_quant:
        print("⚠️ WARNING: quantization_config is still present.")
    else:
        print("Config is clean ✅")

    print("\nAdapter contents:")
    for item in sorted(os.listdir(ADAPTER_SAVE_PATH)):
        print("-", item)

    print("\nMerged contents:")
    for item in sorted(os.listdir(MERGED_SAVE_PATH)):
        size_mb = os.path.getsize(os.path.join(MERGED_SAVE_PATH, item)) / (1024 ** 2)
        print(f"- {item} ({size_mb:.1f} MB)")


Adapter path: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_adapter
Merged model path: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_merged

Saving LoRA adapter...
Adapter + tokenizer saved ✅

Merging LoRA into base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


LoRA merged ✅
Removing quantization_config from merged model config ✅

Saving clean merged model with safetensors shards...


Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

Merged model + tokenizer saved ✅

Verification — quantization_config in config.json: False
Config is clean ✅

Adapter contents:
- README.md
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- tokenizer.json
- tokenizer_config.json

Merged contents:
- chat_template.jinja (0.0 MB)
- config.json (0.0 MB)
- generation_config.json (0.0 MB)
- model-00001-of-00003.safetensors (2079.0 MB)
- model-00002-of-00003.safetensors (3799.7 MB)
- model-00003-of-00003.safetensors (1491.2 MB)
- model.safetensors.index.json (0.1 MB)
- tokenizer.json (10.9 MB)
- tokenizer_config.json (0.0 MB)


## 14. Export Verification for Deployment

**What this cell does:** verifies that the merged export is portable. By default it checks the saved config and tokenizer without loading the full 7B model on CPU, because that can crash low-RAM Colab/Kaggle sessions.

To do a real CPU load test inside the notebook, set `RUN_CPU_LOAD_TEST = True` in Cell 0 before running full training.


In [15]:
# ============================================================
# 14) Export Verification for Deployment
# ============================================================

if RUN_MODE == "smoke":
    print("Skipped on smoke run.")
else:
    from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

    print("Checking merged export:", MERGED_SAVE_PATH)

    config_path = os.path.join(MERGED_SAVE_PATH, "config.json")
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Missing config.json at {config_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        verify_config = json.load(f)

    if "quantization_config" in verify_config:
        raise ValueError("quantization_config still exists in config.json. Do not download yet; fix export first.")

    _ = AutoConfig.from_pretrained(MERGED_SAVE_PATH, trust_remote_code=True)
    _ = AutoTokenizer.from_pretrained(MERGED_SAVE_PATH, trust_remote_code=True)
    print("Config + tokenizer load without bitsandbytes ✅")

    safetensor_files = [f for f in os.listdir(MERGED_SAVE_PATH) if f.endswith(".safetensors")]
    if not safetensor_files:
        raise FileNotFoundError("No .safetensors files found in merged export.")
    print(f"Found {len(safetensor_files)} safetensors file(s) ✅")

    if RUN_CPU_LOAD_TEST:
        print("\nRUN_CPU_LOAD_TEST=True — attempting full CPU load. This may need 16GB+ RAM.")
        test_model = AutoModelForCausalLM.from_pretrained(
            MERGED_SAVE_PATH,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            device_map="cpu",
            low_cpu_mem_usage=True
        )
        test_inputs = tokenizer("Test prompt:", return_tensors="pt")
        with torch.no_grad():
            _ = test_model.generate(**test_inputs, max_new_tokens=5, pad_token_id=tokenizer.pad_token_id)
        del test_model, test_inputs
        torch.cuda.empty_cache()
        print("Full CPU load test passed ✅")
    else:
        print("Full CPU load test skipped by default to avoid crashing low-RAM sessions.")
        print("Set RUN_CPU_LOAD_TEST=True in Cell 0 only if you intentionally want to test CPU loading here.")


Checking merged export: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_merged
Config + tokenizer load without bitsandbytes ✅
Found 3 safetensors file(s) ✅
Full CPU load test skipped by default to avoid crashing low-RAM sessions.
Set RUN_CPU_LOAD_TEST=True in Cell 0 only if you intentionally want to test CPU loading here.


## 15. Evaluation Helpers

**What this cell does:** defines the evaluation functions used for the test set:
- `normalize_text` for fair text comparison (also strips `<think>` blocks for DeepSeek)
- `extract_final_answer` to pull just the final answer line
- `extract_first_number` for arithmetic accuracy
- `token_f1` for partial-match scoring on span/multi-span
- `format_compliance` checks structure correctness per question type
- `prompt_leakage` flags the v1 prompt-leak bug
- `unsupported_style_flag` flags ungrounded financial commentary

**Expected output:** functions defined, no actual computation yet.


In [16]:
# ============================================================
# 15) Evaluation Helpers
# ============================================================

import re
from collections import Counter

def normalize_text(text):
    text = str(text).strip().lower()

    # Strip stop token
    text = text.replace(MODEL_STOP_TOKEN.lower(), "")

    # DeepSeek: strip any leftover <think>...</think> blocks
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*$", "", text, flags=re.DOTALL)

    # Strip leading bullet/hyphen
    text = re.sub(r"^\s*[-\u2022]\s*", "", text)

    # Unify percent representations
    text = re.sub(r"(\d+\.?\d*)\s*per\s*cent\b", r"\1%", text)
    text = re.sub(r"(\d+\.?\d*)\s*percent\b", r"\1%", text)

    # Drop currency symbols
    text = re.sub(r"[$\u00a3\u20ac]", "", text)

    # Drop currency words
    text = re.sub(r"\b(dollars|dollar|pounds|pound|euros|euro|usd|gbp|eur|sar|riyals|riyal)\b", "", text)

    # Round percentages to 1 decimal
    text = re.sub(r"(\d+\.\d)\d+%", r"\1%", text)

    # Drop common filler start
    text = re.sub(r"^the\s+", "", text)

    # Drop trailing period
    text = re.sub(r"\.\s*$", "", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)

    # Keep only useful chars
    text = re.sub(r"[^\w\s.\-%]", "", text)

    return text.strip()


def extract_final_answer(text):
    """Extract Final Answer from gold_full (target_text).
    FIXED: use .* instead of .? so <think> blocks are fully stripped.
    """
    text = str(text)

    # Strip stop token first
    if MODEL_STOP_TOKEN in text:
        text = text.split(MODEL_STOP_TOKEN)[0].strip()

    # Strip <think> blocks — FIXED: .* not .?
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<think>.*$', '', text, flags=re.DOTALL)

    patterns = [
        r'Final Answer[:\s]+(.*?)(?:\n\n|\Z)',
        r'final answer[:\s]+(.*?)(?:\n\n|\Z)',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            return m.group(1).strip()

    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    return lines[-1] if lines else text.strip()


def extract_first_number(text):
    text = str(text).replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", text)
    if match:
        try:
            return float(match.group())
        except Exception:
            return None
    return None


def extract_pred_final_deepseek(text):
    """Extract Final Answer from pred_full (model output)."""
    text = str(text)

    if MODEL_STOP_TOKEN in text:
        text = text.split(MODEL_STOP_TOKEN)[0].strip()

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*$", "", text, flags=re.DOTALL)

    m = re.search(
        r"Final Answer[:\s]+(.*?)(?:\n\n|\Z)",
        text,
        re.IGNORECASE | re.DOTALL
    )
    if m:
        return m.group(1).strip()

    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    return lines[-1] if lines else text.strip()


def token_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    pred_counter = Counter(pred_tokens)
    gold_counter = Counter(gold_tokens)
    overlap = sum((pred_counter & gold_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall    = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def section_order_ok(text):
    text = str(text)
    sections = [
        "Reasoning & Calculations:",
        "Values used:",
        "Calculation:",
        "Analysis:",
        "Final Answer:"
    ]
    positions = []
    for s in sections:
        pos = text.find(s)
        if pos == -1:
            return False
        positions.append(pos)
    return positions == sorted(positions)


def format_compliance(text, answer_type):
    text = str(text)
    answer_type = str(answer_type).lower()
    base_ok = (
        "Reasoning & Calculations:" in text and
        "Values used:" in text and
        "Calculation:" in text and
        "Analysis:" in text and
        "Final Answer:" in text
    )
    if not base_ok:
        return 0
    if not section_order_ok(text):
        return 0
    if answer_type == "arithmetic":
        return 1
    return int("No financial analysis required" in text)


def prompt_leakage(text):
    text = str(text).lower()
    leaks = [
        "you are an expert financial analyst",
        "use only the provided",
        "structure your response",
        "human:",
        "assistant:",
        "question:",
        "table context:",
        "paragraph context:"
    ]
    return int(any(x in text for x in leaks))


def unsupported_style_flag(text):
    text = str(text).lower()
    bad_phrases = [
        "to verify",
        "further analysis",
        "would be necessary",
        "market share",
        "creditworthiness",
        "future prospects",
        "recommend",
        "should continue"
    ]
    return int(any(p in text for p in bad_phrases))


print("Evaluation helpers defined ✅")


Evaluation helpers defined ✅


In [17]:
print("exists?", "extract_pred_final_deepseek" in globals())
print(extract_pred_final_deepseek)

exists? True
<function extract_pred_final_deepseek at 0x78092814fa60>


## 16. Full Test Set Evaluation

**What this cell does:** runs full inference evaluation only when `RUN_MODE = "full"`.

**Smoke mode:** skipped on purpose. The 5 manual cases are enough for smoke validation.

**Full mode expected output:** `results_df` with predictions and metrics for the whole test set.


In [18]:
# ============================================================
# 16) Full Test Set Evaluation
# ============================================================

if RUN_MODE == "smoke":
    print("Skipped on smoke run — the 5 manual cases in Cell 12 are enough for smoke validation.")
    results_df = pd.DataFrame()
    eval_time_seconds = 0.0
    results_path = None
else:
    eval_rows = []
    start_eval = time.time()

    for idx, row in test_df.iterrows():
        prompt = row["input_text"]
        gold_full = row["target_text"]
        gold_final = extract_final_answer(gold_full)

        pred_full = generate_response(prompt, max_new_tokens=400)
        pred_final = extract_pred_final_deepseek(pred_full)

        answer_type = str(row["answer_type"]).strip().lower()

        em  = int(normalize_text(pred_final) == normalize_text(gold_final))
        f1  = token_f1(pred_final, gold_final)
        fmt = format_compliance(pred_full, answer_type)

        gold_num = extract_first_number(gold_final)
        pred_num = extract_first_number(pred_final)

        numeric_exact      = np.nan
        numeric_close_1pct = np.nan
        relative_error     = np.nan

        if answer_type == "arithmetic":
            if gold_num is not None and pred_num is not None:
                numeric_exact = int(gold_num == pred_num)
                if gold_num != 0:
                    relative_error = abs(pred_num - gold_num) / abs(gold_num)
                else:
                    relative_error = 0.0 if pred_num == 0 else np.nan
                numeric_close_1pct = int(not np.isnan(relative_error) and relative_error <= 0.01)
            else:
                numeric_exact      = 0
                numeric_close_1pct = 0

        eval_rows.append({
            "idx":                 idx,
            "question":            row["question"],
            "answer_type":         answer_type,
            "gold_final":          gold_final,
            "pred_final":          pred_final,
            "pred_full":           pred_full,
            "exact_match":         em,
            "f1":                  f1,
            "format_compliance":   fmt,
            "prompt_leakage":      prompt_leakage(pred_full),
            "unsupported_style":   unsupported_style_flag(pred_full),
            "gold_num":            gold_num,
            "pred_num":            pred_num,
            "numeric_exact":       numeric_exact,
            "numeric_close_1pct":  numeric_close_1pct,
            "relative_error":      relative_error,
            "pred_tokens":         len(tokenizer(pred_full, add_special_tokens=False)["input_ids"])
        })

    end_eval = time.time()
    eval_time_seconds = end_eval - start_eval

    results_df = pd.DataFrame(eval_rows)

    print("Evaluation completed ✅")
    print("Evaluation time seconds:",   round(eval_time_seconds, 2))
    print("Average time per sample:",   round(eval_time_seconds / len(test_df), 2))

    display(results_df.head(10))


Evaluation completed ✅
Evaluation time seconds: 11255.58
Average time per sample: 32.16


,idx,question,answer_type,gold_final,pred_final,pred_full,exact_match,f1,format_compliance,prompt_leakage,unsupported_style,gold_num,pred_num,numeric_exact,numeric_close_1pct,relative_error,pred_tokens
0,0,What does Income (loss) before expense (benefi...,span,Income (loss) before expense (benefit) for inc...,should clearly state what it reflects.,"Okay, so I need to figure out what ""Income (lo...",0,0.074074,0,0,0,NaN,NaN,NaN,NaN,NaN,400
1,1,What was the percentage change in the Amortiza...,arithmetic,The percentage change in the Amortization of c...,"Wait, but the question is about the percentage...","Okay, so I need to figure out the percentage c...",0,0.222222,0,0,0,2018.0,NaN,0.0,0.0,NaN,400
2,2,Where was receivables from related parties inc...,span,Receivables from related parties were included...,Receivables from related parties were included...,"Okay, so I need to figure out where ""Receivabl...",0,0.782609,0,0,0,NaN,NaN,NaN,NaN,NaN,365
3,3,What is the sum of the weighted average shares...,arithmetic,The sum of the weighted average shares outstan...,"should be 170,658 shares.\n</think>","Okay, so I need to figure out the sum of the w...",0,0.000000,0,0,1,2018.0,170658.000,0.0,0.0,83.567889,345
4,4,What was the average Adjusted EBITDA for 2018 ...,arithmetic,The average Adjusted EBITDA for 2018 and 2019 ...,"with the appropriate scale, which is in millio...","Alright, so I need to figure out the average A...",0,0.000000,0,0,0,2018.0,NaN,0.0,0.0,NaN,400
5,5,What is the Tax expense at U.S. statutory rate...,span,The Tax expense at U.S. statutory rate for 201...,The Tax expense at U.S. statutory rate for 201...,"Okay, so I need to figure out the Tax expense ...",0,0.769231,0,0,0,2019.0,2019.000,NaN,NaN,NaN,360
6,6,What was the adjusted EBITDA in 2019?,span,"The adjusted EBITDA in 2019 was $108,307 thous...","should state the value, mention that it's in t...","Okay, so I need to figure out the adjusted EBI...",0,0.064516,0,0,1,2019.0,NaN,NaN,NaN,NaN,399
7,7,Which years does the table provide information...,multi-span,The table provides information for accrued exp...,should be a single sentence stating the years....,"Okay, so I need to figure out which years the ...",0,0.083333,0,0,0,2020.0,NaN,NaN,NaN,NaN,308
8,8,What is the increase/ (decrease) in Amortized ...,arithmetic,The increase in Amortized Cost of U.S. Treasur...,should be that the Amortized Cost increased by...,"Okay, so I need to figure out the increase or ...",0,0.121212,0,0,0,28.0,2.461,0.0,0.0,0.912107,400
9,9,What is the effective tax rate in the year 201...,multi-span,"The effective tax rate is 52.8% in 2017, 9.7% ...",The effective tax rates for the years 20,"Okay, so I need to figure out the effective ta...",0,0.181818,0,0,0,52.8,20.000,NaN,NaN,NaN,400


## 17. Summary Metrics for Comparison

**What this cell does:** aggregates full test-set results into summary CSVs for the paper/comparison table — directly comparable to the Qwen and Llama summary CSVs.

**Smoke mode:** skipped on purpose because full metrics require the full evaluation.


In [19]:
# ============================================================
# 17) Summary Metrics for Comparison
# ============================================================

if RUN_MODE == "smoke":
    print("Skipped on smoke run — summary metrics require the full test-set evaluation.")
    summary_df = pd.DataFrame()
    metrics_by_type = pd.DataFrame()
    results_path = None
    summary_path = None
    by_type_path = None
else:
    arithmetic_df     = results_df[results_df["answer_type"] == "arithmetic"]
    non_arithmetic_df = results_df[results_df["answer_type"] != "arithmetic"]

    summary = {
        "model":                          MODEL_NAME,
        "model_tag":                      MODEL_TAG,
        "run_mode":                       RUN_MODE,
        "train_samples":                  len(train_df_used),
        "val_samples":                    len(val_df_used),
        "test_samples":                   len(test_df),
        "epochs":                         num_epochs,
        "max_length":                     MAX_LENGTH,
        "effective_batch_size":           effective_batch_size,
        "learning_rate":                  training_args.learning_rate,
        "trainable_params":               trainable_params,
        "all_params":                     all_params,
        "trainable_pct":                  trainable_pct,
        "training_time_seconds":          training_time_seconds,
        "eval_time_seconds":              eval_time_seconds,
        "avg_inference_time_seconds":     eval_time_seconds / len(test_df),
        "overall_em":                     results_df["exact_match"].mean(),
        "overall_f1":                     results_df["f1"].mean(),
        "overall_format_compliance":      results_df["format_compliance"].mean(),
        "prompt_leakage_rate":            results_df["prompt_leakage"].mean(),
        "unsupported_style_rate":         results_df["unsupported_style"].mean(),
        "avg_pred_tokens":                results_df["pred_tokens"].mean(),
        "non_arithmetic_em":              non_arithmetic_df["exact_match"].mean() if len(non_arithmetic_df) else np.nan,
        "non_arithmetic_f1":              non_arithmetic_df["f1"].mean() if len(non_arithmetic_df) else np.nan,
        "arithmetic_numeric_exact":       arithmetic_df["numeric_exact"].dropna().mean() if len(arithmetic_df) else np.nan,
        "arithmetic_numeric_close_1pct":  arithmetic_df["numeric_close_1pct"].dropna().mean() if len(arithmetic_df) else np.nan,
        "arithmetic_mean_relative_error": arithmetic_df["relative_error"].dropna().mean() if len(arithmetic_df) else np.nan,
    }

    summary_df = pd.DataFrame([summary])

    metrics_by_type = results_df.groupby("answer_type").agg(
        samples=("idx", "count"),
        exact_match=("exact_match", "mean"),
        f1=("f1", "mean"),
        format_compliance=("format_compliance", "mean"),
        prompt_leakage=("prompt_leakage", "mean"),
        unsupported_style=("unsupported_style", "mean"),
        avg_pred_tokens=("pred_tokens", "mean")
    ).reset_index()

    print("===== SUMMARY METRICS =====")
    display(summary_df)

    print("===== METRICS BY ANSWER TYPE =====")
    display(metrics_by_type)

    results_path = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_test_predictions.csv"
    summary_path = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_summary_metrics.csv"
    by_type_path = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_metrics_by_type.csv"

    results_df.to_csv(results_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    metrics_by_type.to_csv(by_type_path, index=False)

    print("Saved:")
    print(results_path)
    print(summary_path)
    print(by_type_path)


===== SUMMARY METRICS =====


,model,model_tag,run_mode,train_samples,val_samples,test_samples,epochs,max_length,effective_batch_size,learning_rate,...,overall_f1,overall_format_compliance,prompt_leakage_rate,unsupported_style_rate,avg_pred_tokens,non_arithmetic_em,non_arithmetic_f1,arithmetic_numeric_exact,arithmetic_numeric_close_1pct,arithmetic_mean_relative_error
0,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,deepseek_r1_distill_qwen_7b,full,2798,350,350,2,1024,16,0.0002,...,0.320009,0.014286,0.0,0.062857,382.614286,0.0,0.35747,0.142857,0.204082,1039.385335


===== METRICS BY ANSWER TYPE =====


,answer_type,samples,exact_match,f1,format_compliance,prompt_leakage,unsupported_style,avg_pred_tokens
0,arithmetic,147,0.0,0.268276,0.034014,0.0,0.088435,392.210884
1,count,8,0.0,0.426236,0.000000,0.0,0.000000,366.125000
2,multi-span,44,0.0,0.349482,0.000000,0.0,0.022727,372.704545
3,span,151,0.0,0.356154,0.000000,0.0,0.052980,377.033113


Saved:
/content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_test_predictions.csv
/content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_summary_metrics.csv
/content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_metrics_by_type.csv


## 18. Model Size + Deployment Readiness

**What this cell does:** checks adapter and merged model sizes after full export and saves a deployment-readiness CSV.

**Smoke mode:** skipped on purpose.


In [20]:
# ============================================================
# 18) Model Size Check
# ============================================================

if RUN_MODE == "smoke":
    print("Skipped on smoke run — adapter/merged export only happens in full mode.")
    cpu_readiness_path = None
    cpu_readiness = pd.DataFrame()
else:
    def get_folder_size_gb(folder_path):
        total_size = 0
        for dirpath, dirnames, filenames in os.walk(folder_path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.exists(fp):
                    total_size += os.path.getsize(fp)
        return total_size / (1024 ** 3)

    adapter_size_gb = get_folder_size_gb(ADAPTER_SAVE_PATH) if os.path.exists(ADAPTER_SAVE_PATH) else np.nan
    merged_size_gb  = get_folder_size_gb(MERGED_SAVE_PATH)  if os.path.exists(MERGED_SAVE_PATH)  else np.nan

    cpu_readiness = pd.DataFrame([{
        "model":           MODEL_NAME,
        "adapter_path":    ADAPTER_SAVE_PATH,
        "adapter_size_gb": adapter_size_gb,
        "merged_path":     MERGED_SAVE_PATH,
        "merged_size_gb":  merged_size_gb,
        "cpu_note":        "Merged model exported with clean config and safetensors shards. For practical CPU inference, consider GGUF/llama.cpp conversion if supported."
    }])

    display(cpu_readiness)

    cpu_readiness_path = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_cpu_readiness.csv"
    cpu_readiness.to_csv(cpu_readiness_path, index=False)

    print("Saved:", cpu_readiness_path)


,model,adapter_path,adapter_size_gb,merged_path,merged_size_gb,cpu_note
0,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,/content/drive/MyDrive/gp2_training/deepseek_r...,0.161085,/content/drive/MyDrive/gp2_training/deepseek_r...,7.207934,Merged model exported with clean config and sa...


Saved: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_cpu_readiness.csv


## 19. Zip Files for Download

**What this cell does:** zips the adapter, merged model, and result CSVs in full mode.

**Colab:** zip files are saved in Google Drive under `gp2_training/<MODEL_TAG>`.

**Kaggle:** zip files are also copied to `/kaggle/working/output/` so they show in the Output tab after you save/commit the notebook version.

**Smoke mode:** skipped on purpose.


In [21]:
# ============================================================
# 19) Zip Files for Download / Kaggle Output
# ============================================================

if RUN_MODE == "smoke":
    print("Skipped on smoke run — no adapter/merged zip needed.")
else:
    zip_adapter_path = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_adapter.zip"
    zip_merged_path  = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_merged.zip"
    zip_results_path = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_results.zip"

    if os.path.exists(ADAPTER_SAVE_PATH):
        shutil.make_archive(zip_adapter_path.replace(".zip", ""), "zip", ADAPTER_SAVE_PATH)
        print("Adapter zip:", zip_adapter_path)

    if os.path.exists(MERGED_SAVE_PATH):
        shutil.make_archive(zip_merged_path.replace(".zip", ""), "zip", MERGED_SAVE_PATH)
        print("Merged zip:", zip_merged_path)

    results_files = []
    for var_name in [
        "training_log_path",
        "training_metrics_path",
        "results_path",
        "summary_path",
        "by_type_path",
        "cpu_readiness_path"
    ]:
        value = globals().get(var_name)
        if value and os.path.exists(value):
            results_files.append(value)

    results_folder = f"{PERSISTENT_DIR}/{MODEL_TAG}_{RUN_MODE}_results_bundle"
    os.makedirs(results_folder, exist_ok=True)

    for f in results_files:
        shutil.copy(f, results_folder)

    shutil.make_archive(zip_results_path.replace(".zip", ""), "zip", results_folder)
    print("Results zip:", zip_results_path)

    # Kaggle only: copy final zips to /kaggle/working/output so they appear in the Output tab.
    if IS_KAGGLE:
        kaggle_output_dir = "/kaggle/working/output"
        os.makedirs(kaggle_output_dir, exist_ok=True)
        for z in [zip_adapter_path, zip_merged_path, zip_results_path]:
            if os.path.exists(z):
                target = os.path.join(kaggle_output_dir, os.path.basename(z))
                shutil.copy(z, target)
                print("Copied to Kaggle output:", target)

        print("\nKaggle download instructions:")
        print("1. Click Save Version / Commit in the top right.")
        print("2. Wait for the version to finish saving.")
        print("3. Open the Output tab.")
        print("4. Download the zip files from /kaggle/working/output/.")
    elif IS_COLAB:
        print("\nColab download note:")
        print("The zip files are saved in Google Drive under:", PERSISTENT_DIR)
        print("You can download them from Drive directly.")


Adapter zip: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_adapter.zip
Merged zip: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_merged.zip
Results zip: /content/drive/MyDrive/gp2_training/deepseek_r1_distill_qwen_7b/deepseek_r1_distill_qwen_7b_full_results.zip
Copied to Kaggle output: /kaggle/working/output/deepseek_r1_distill_qwen_7b_full_adapter.zip
Copied to Kaggle output: /kaggle/working/output/deepseek_r1_distill_qwen_7b_full_merged.zip
Copied to Kaggle output: /kaggle/working/output/deepseek_r1_distill_qwen_7b_full_results.zip

Kaggle download instructions:
1. Click Save Version / Commit in the top right.
2. Wait for the version to finish saving.
3. Open the Output tab.
4. Download the zip files from /kaggle/working/output/.
